In [1]:
import os

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["PINECONE_API_KEY"]=os.getenv("PINECONE_API_KEY")
os.environ["HF_API_KEY"]=os.getenv("HF_API_KEY")


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader("../data/hai_ai_index_report_2025.pdf")
doc=loader.load()


c:\Users\Lenovo\Desktop\Himesh_folder\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
doc[5].page_content

'Artificial Intelligence\nIndex Report 2025\n5\nTop Takeaways (cont’d)\n11. AI earns top honors for its impact on science.  AI’s growing importance is reflected in major scientific awards: \nTwo Nobel Prizes recognized work that led to deep learning (physics) and to its application to protein folding (chemistry), \nwhile the Turing Award honored groundbreaking contributions to reinforcement learning.\n12. Complex reasoning remains a challenge.  AI models excel at tasks like International Mathematical Olympiad \nproblems but still struggle with complex reasoning benchmarks like PlanBench. They often fail to reliably solve logic tasks even \nwhen provably correct solutions exist, limiting their effectiveness in high-stakes settings where precision is critical.'

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
spliter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)
finaldoc=spliter.split_documents(doc)
finaldoc

[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 20.5 (Macintosh)', 'creationdate': '2025-11-10T12:19:12-08:00', 'author': 'Nestor Maslej', 'moddate': '2025-11-12T13:18:48-08:00', 'subject': 'Welcome to the eighth edition of the AI Index report. The 2025 Index is our most comprehensive to date and arrives at an important moment, as AI’s influence across society, the economy, and global governance continues to intensify. New in this year’s report are in-depth analyses of the evolving landscape of AI hardware, novel estimates of inference costs, and new analyses of AI publication and patenting trends. We also introduce fresh data on corporate adoption of responsible AI practices, along with expanded coverage of AI’s growing role in science and medicine. Since its founding in 2017 as an offshoot of the One Hundred Year Study of Artificial Intelligence, the AI Index has been committed to equipping policymakers, journalists, executives, researchers, and t

In [19]:
text=[doc.page_content for doc in finaldoc]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7079.40it/s]


330

In [6]:
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


vectorstore = PineconeVectorStore.from_documents(
    documents=finaldoc,
    embedding=embedding,
    index_name="himesh",
    pinecone_api_key="PINECONE_API_KEY"
)

print("Stored successfully!")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3598.76it/s]


Stored successfully!


In [15]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# User query
query = "what is ai"

# Retrieve relevant docs
docs = retriever.invoke(query)

# Convert docs to context
context = "\n\n".join([doc.page_content for doc in docs])

print(context)

337
Artificial Intelligence
Index Report 2025Table of Contents Chapter 6 Preview
6.2 AI and Policymaking
Global Legislative Records on AI
Overview
The AI Index analyzed legislation containing the term 
“artificial intelligence” in 114 countries from 2016 to 2024. 1 
Of these, 39 countries have enacted at least one AI-related 
law (Figure 6.2.1).
2 In total, the countries have passed 204 
AI-related laws. Figure 6.2.2 illustrates the annual count of

246
Artificial Intelligence
Index Report 2025Table of Contents Chapter 4 Preview
Highlight:  
Measuring AI’s Current Economic Integration (cont’d)
4.2 Jobs
Chapter 4: Economy
The analysis reveals how AI is being used within 
organizations. As shown in Figure 4.2.27, 57% of AI 
interactions demonstrate augmentative patterns 
(enhancing human capabilities) while 43% show 
automation patterns. This split suggests current AI 
implementation tends toward complementing rather than

453
Artificial Intelligence
Index Report 2025Appendix
Table of Co

In [25]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
llm=ChatGroq(
    model="llama-3.1-8b-instant",
    groq_api_key=os.getenv("GROQ_API_KEY")
)

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful assistant.
Answer only from the provided context.

Context:
{context}

Question:
{question}

Answer:
"""
)

# Create chain
chain = prompt|llm

# Generate answer
response = chain.invoke({
    "context": context,
    "question": query
})

print(response.content)


Artificial Intelligence, or AI, refers to the use of computer systems to perform tasks that would typically require human intelligence, such as learning, problem-solving, decision-making, and perception.
